In [ ]:
import pandas as pd 
import numpy as np 

#### Business Goal - Increase revenue

Are sales growing every year?

In [9]:
# Total orders, sales, products, quantities through 2014-2017
orders_rev = (orders_agg.groupby(["year"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year"])["products"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["quantity"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["revenue"].sum(), 
    how="left", 
    on="year")
orders_rev["%_change"] = round(orders_rev["revenue"].pct_change(),2)
orders_rev

,year,order_id,products,quantity,revenue,%_change
0,2014,969,1992,7579,483966.1261,NaN
1,2015,1038,2102,7979,470532.5090,-0.03
2,2016,1315,2587,9837,609205.5980,0.29
3,2017,1687,3312,12476,733215.2552,0.20


There has been a steady increase in number of orders placed, products sold and quantities. However, if you look at revenue, it decreased in 2015 by 3% but experienced a ~30% jump in 2016 and 20% jump in 2017

Yearly-revenue wise things look steady.

Looking at the revenue quarter wise to ensure this does not fall under Simpson's paradox. Hence, going a level deeper

In [10]:
# Total orders, sales, products, quantities: quarterly through 2014-2017
orders_rev_qtr = (orders_agg.groupby(["year", "quarter"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "quarter"])["products"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["quantity"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["revenue"].sum(), 
    how="left", 
    on=["year", "quarter"])
orders_rev_qtr["%_change"] = round(orders_rev_qtr["revenue"].pct_change(),2)
orders_rev_qtr

,year,quarter,order_id,products,quantity,revenue,%_change
0,2014,1,185,385,1438,96498.7200,NaN
1,2014,2,208,405,1516,83355.5086,-0.14
2,2014,3,257,545,2097,139306.0173,0.67
3,2014,4,319,657,2528,164805.8802,0.18
4,2015,1,186,342,1301,90952.3496,-0.45
5,2015,2,248,488,1788,97852.8812,0.08
6,2015,3,275,588,2273,145554.2330,0.49
7,2015,4,329,684,2617,136173.0452,-0.06
8,2016,1,235,473,1782,136898.6390,0.01
9,2016,2,308,637,2394,149148.5428,0.09


In [11]:
print("Typical increase in revenue each quarter", round(orders_rev_qtr["%_change"].median(),2))

Typical increase in revenue each quarter 0.04


There seems to have been significant drop in revenue every third quarter as a repeating cycle through the years in comparison to the typical expected increase of ~ 4% every quarter.

Digging a level deeper - Month wise

In [12]:
# Total orders, sales, products, quantities: monthly through 2014-2017
orders_rev_month = round((orders_agg.groupby(["year", "month_num", "month"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "month_num", "month"])["products"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["quantity"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["revenue"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]),2).sort_values(["month_num", "year"])

orders_rev_month_pct_change = orders_rev_month.pivot(
    index=["month_num","month"],
    columns = "year",
    values = "revenue"
).reset_index()
orders_rev_month_pct_change.columns = ["month_num", "month", "2014", "2015", "2016", "2017"]
orders_rev_month_pct_change.index = orders_rev_month_pct_change["month"]
round(orders_rev_month_pct_change[[ "2014", "2015", "2016", "2017"]].pct_change(axis=1)*100,2)


,2014,2015,2016,2017
month,,,,
January,NaN,1.36,29.65,70.14
February,NaN,62.66,137.54,1.57
March,NaN,-25.41,21.37,50.72
April,NaN,55.79,18.75,-13.54
May,NaN,4.37,110.01,-37.07
June,NaN,-1.45,35.10,22.44
July,NaN,-18.71,48.88,27.14
August,NaN,32.33,-7.49,63.30
September,NaN,0.94,-37.08,76.64


In [13]:
print("Typically the order value for each year is: \n", (round(orders_agg.groupby(
    ["year"])["revenue"].median(),2)))

Typically the order value for each year is: 
 year
2014    155.37
2015    162.07
2016    145.50
2017    148.26
Name: revenue, dtype: float64


After digging a level deeper, it was observed that -on the products, quantities and orders level it seems like a steady growth.
 
And, the typical* order value for each year has seen a slight drop. Indicating the company is selling more (in reference to orders placed, quantities & products sold had been steadily increasing) but still earning less revenue per order.

*Median order value was used instead of average order value to reduce the influence of unusually large orders

#### Pareto Analysis

In [ ]:
pareto_product_analysis = pd.DataFrame(round(orders.groupby("Product ID")["Sales"].sum(),2).sort_values(ascending = False).reset_index())
pareto_product_analysis["cumulative_revenue"] = round(pareto_product_analysis["Sales"].cumsum(), 2)
pareto_product_analysis["cumulative_%"] = round((pareto_product_analysis["cumulative_revenue"]/
                                           pareto_product_analysis["Sales"].sum())*100,2)
pareto_product_analysis[pareto_product_analysis["cumulative_%"]<=80]


It can be observed that around 22% of top products (based on revenue) contribute to 80% of revenue